# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [1]:
%pip install -Uqqq datasets langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [3]:
from datasets import load_dataset

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')
dataset  # train / valid / test

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [4]:
# 필터링 : train / category = 'economy'인 것만 남김
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))

17088


In [5]:
economy_dataset[0]

{'date': '2022-07-03 17:14:37',
 'category': 'economy',
 'press': 'YTN ',
 'title': '추경호 중기 수출지원 총력 무역금융 40조 확대',
 'document': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 'link': 'https://n.news.naver.com/mne

In [6]:
import pandas as pd

df = economy_dataset.to_pandas()  # Dataset -> Pandas DataFrame
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변데이터 생성
llm을 이용해서 sLLM이 답변했으면 하는 내용을 생성해낸다. 이때 답변을 품질이 중요하므로, 되도록 상위모델을 사용하는 것이 좋다.

In [7]:
# 분석용 출력 스키마(Pydantic) + 프롬프트 템플릿 구성
from pydantic import BaseModel, Field  # 구조화 출력 스키마 정의
from typing import List, Optional      # 타입 힌트
from langchain_core.prompts import ChatPromptTemplate  # 채팅 프롬프트 템플릿

# 금융뉴스 분석결과 출력 클래스
class StockAnalysis(BaseModel):
    stock_related: bool = Field(description="뉴스와 주식 종목간의 연관성 여부")  # 종목 연관성 (True/Flse)
    summary: str = Field(description='뉴스 요약')
    
    # 수혜 종목 리스트 / 근거 키워드 / 이유
    positive_stocks: List[str] = Field(description='긍정적인 영향이 예상되는 주식 종목명 목록')
    positive_keywords: List[str] = Field(description='긍정적인 영향의 근거가 되는 키워드 목록')
    positive_reasons: str = Field(description='긍정적인 영향이 예상되는 이유')
    
    # 피해 종목 리스트 / 근거 키워드 / 이유
    negative_stocks: List[str] = Field(description='부적인 영향이 예상되는 주식 종목명 목록')
    negative_keywords: List[str] = Field(description='부정적인 영향의 근거가 되는 키워드 목록')
    negative_reasons: str = Field(description='부정적인 영향이 예상되는 이유')

system_prompt = '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', user_prompt),
])

news = df['document'][0]  # 첫 번째 뉴스
prompt.invoke({'news': news})  # 프롬프트에 내용 채워 확인

ChatPromptValue(messages=[SystemMessage(content="\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='\n다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추

In [8]:
# Lanchain 구성 및 구조화 출력
from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-5.6-luna')
chain = prompt | llm.with_structured_output(StockAnalysis)  # 프롬프트 -> LLM -> StockAnalysis 구조화 출력

# 뉴스 본문 기반으로 구조화된 분석 결과 반환하는 함수
def analyze_news(news):
    return chain.invoke({'news': news})

analyze_news(news)

StockAnalysis(stock_related=True, summary='정부는 하반기 수출 증가세를 유지하고 무역수지를 개선하기 위해 수출 중소·중견기업 대상 무역금융을 연초 목표보다 40조원 늘린 301조원까지 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대를 추진한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 방안을 마련할 계획이다. 이는 상반기 수출 호조에도 원자재 가격 상승과 수입 급증으로 무역수지가 103억달러 적자를 기록한 데 따른 대응책이다.', positive_stocks=['삼성전자', 'SK하이닉스', '현대차', '기아', 'LG전자', 'HMM'], positive_keywords=['무역금융 301조원 확대', '수출 중소·중견기업 지원', '물류비 지원', '임시선박 투입', '반도체 등 첨단산업 육성', '해외 전시회 마케팅 지원'], positive_reasons='무역금융 확대와 물류비 지원은 수출기업의 자금 조달 및 운송 부담을 낮춰 수출 물량과 이익 방어에 기여할 수 있다. 반도체 육성 전략은 삼성전자와 SK하이닉스 등 국내 대표 반도체 기업의 설비·기술 투자와 수출 경쟁력에 우호적이다. 자동차·가전 등 주요 수출 기업도 금융·물류 지원과 해외 마케팅 확대의 수혜가 가능하다. HMM은 임시선박 운항 및 선복 확대 과정에서 수출 물동량 증가의 수혜를 받을 수 있으나, 선복 공급 확대가 운임 하락으로 이어질 경우 수익성에는 제한적일 수 있다.', negative_stocks=[], negative_keywords=[], negative_reasons='')

In [9]:
news = df['document'][100]
display(news)  # Jupyter / Colab용 화면출력
print()

analyze_news(news)

'해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.'

StockAnalysis(stock_related=True, summary='해양수산부가 공유수면 점용·사용 허가 과정에서 어업인 등 이해관계자의 사전 의견 수렴을 의무화하는 개정 공유수면법과 시행령·시행규칙을 시행했다. 해양환경·수산자원·자연경관 등에 영향을 줄 수 있는 신청은 관보·공보 및 인터넷에 공고해야 하며, 피해가 예상되는 어업인에 대한 별도 의견조사도 진행된다. 이번 개정은 해상풍력, 해변 관광시설 등 대규모·장기 공유수면 이용사업에서 발생해 온 어민·지역사회와의 갈등을 줄이는 것이 목적이다.', positive_stocks=[], positive_keywords=[], positive_reasons='', negative_stocks=['SK오션플랜트', '씨에스윈드', '두산에너빌리티'], negative_keywords=['공유수면 점용·사용 허가', '이해관계자 사전 의견수렴', '어업인 의견조사', '해상풍력 인허가', '사업 지연 및 비용 증가'], negative_reasons='이번 제도 개정은 해상풍력 등 공유수면을 활용하는 대규모 사업의 인허가 절차를 추가하고 이해관계자 협의 요건을 강화한다. 이에 따라 사업 초기의 환경·어업 영향 검토, 공고 및 의견수렴, 보완 협의에 필요한 기간과 비용이 늘어날 수 있으며, 어민·지역사회 반대가 클 경우 착공 및 수주 일정이 지연될 가능성이 있다. SK오션플랜트는 해상풍력 하부구조물, 씨에스윈드는 풍력 타워, 두산에너빌리티는 풍력발전 설비 사업과 연관성이 있어 국내 해상풍력 프로젝트 지연 시 수주 인식 시점이나 국내 매출 확대 속도에 부정적일 수 있다. 다만 법 시행이 기존 사업을 일괄 중단시키는 것은 아니며, 갈등이 조기에 해소되면 장기적으로는 인허가 불확실성이 낮아지는 효과도 가능해 실제 영향은 개별 프로젝트의 진행 단계와 지역 수용성에 따라 달라진다.')

In [10]:
# 뉴스 데이터 1000건 샘플링
df = df[:1000]
df['content'] = df['title'] + '\n' + df['document']  # content = 제목 + 본문

pd.set_option('display.max_colwidth', None)  # 컬럼 내용 짤림방지
df['content'].head()

0                                                                                                                                                                      추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 

In [11]:
# 증류방식 활용 (GPT 답변을 로컬LLM에게 학습)
from tqdm.auto import tqdm

results = []  # 분석 결과 저장 리스트

for content in tqdm(df['content']):
    result = analyze_news(content)
    results.append(result)

df['result'] = results  # 결과 리스트를 df 컬럼으로 추가
df.head()

  0%|          | 5/1000 [00:47<2:39:03,  9.59s/it]


KeyboardInterrupt: 

In [ ]:
# Pydantic(StockAnlysis) 결과를 JSON 문자열로 파싱
def parse_to_json(obj):
    return obj.model_dump_json()  # StockAnlysis -> JSON 문자열 파싱

df['result_json'] = df['result'].apply(parse_to_json)  # df 각 행마다 함수 적용해 파생변수 생성
df.head()

,date,category,press,title,document,link,summary,content,result,result_json
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,https://n.news.naver.com/mnews/article/052/0001759333?sid=101,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다.",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,"stock_related=True summary='정부는 원자재 가격 상승과 물류난 등 대외 리스크에 대응하고 수출 증가세를 유지하기 위해 하반기 수출지원 대책을 추진한다. 수출 중소·중견기업 지원을 위해 무역금융을 당초 계획보다 40조원 늘려 총 301조원 규모로 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대를 시행한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 방안을 마련해 수출 경쟁력과 무역수지 개선을 도모할 계획이다. 정책의 직접 수혜는 수출 중소기업과 첨단산업에 집중될 가능성이 높지만, 무역금융 확대만으로 수입 원자재 가격 상승에 따른 무역수지 적자가 해소된다고 보기는 어렵다.' positive_stocks=['삼성전자', 'SK하이닉스', '현대차', '기아', 'LG전자'] positive_keywords=['무역금융 40조원 확대', '수출 중소·중견기업 지원', '반도체 등 첨단산업 육성', '물류비 지원', '해외 전시회 및 수출 마케팅 지원'] positive_reasons='무역금융 확대는 자금 조달 여력이 제한적인 수출기업의 생산·수주·운전자금 부담을 완화해 수출 물량 확대에 도움을 줄 수 있다. 삼성전자와 SK하이닉스는 정부의 반도체 중심 첨단산업 육성 전략의 직접적인 정책 수혜 가능성이 있으며, 현대차·기아·LG전자 등 글로벌 판매 비중이 높은 기업도 금융·물류 지원에 따른 수출 비용 절감과 해외 마케팅 확대의 간접 수혜가 예상된다. 다만 대형 기업은 중소기업보다 정책 지원의 직접 효과가 제한적이고, 실제 주가 영향은 반도체 수요와 원자재·환율·운임 등 시장 변수에 더 크게 좌우될 수 있다.' negative_stocks=['HMM', '대한해운'] negative_keywords=['임시선박 월 4척 이상 투입', '중소기업 전용 선복 확대', '해상운임 안정화', '선복 공급 증가'] negative_reasons='정부가 임시선박을 지속 투입하고 중소기업 전용 선복을 확대하면 수출기업의 물류난은 완화되지만, 해상운송 공급이 늘어 운임이 하락할 경우 HMM·대한해운과 같은 해운사의 운임 및 수익성에는 부담이 될 수 있다. 다만 수출 물동량 증가가 운임 하락 효과를 상쇄할 가능성이 있어 부정적 영향은 해상운임과 글로벌 물동량의 방향에 따라 달라진다.'","{""stock_related"":true,""summary"":""정부는 원자재 가격 상승과 물류난 등 대외 리스크에 대응하고 수출 증가세를 유지하기 위해 하반기 수출지원 대책을 추진한다. 수출 중소·중견기업 지원을 위해 무역금융을 당초 계획보다 40조원 늘려 총 301조원 규모로 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대를 시행한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 방안을 마련해 수출 경쟁력과 무역수지 개선을 도모할 계획이다. 정책의 직접 수혜는 수출 중소기업과 첨단산업에 집중될 가능성이 높지만, 무역금융 확대만으로 수입 원자재 가격 상승에 따른 무역수지 적자가 해소된다고 보기는 어렵다."",""positive_stocks"":[""삼성전자"",""SK하이닉스"",""현대차"",""기아"",""LG전자""],""positive_keywords"":[""무역금융 40조원 확대"",""수출 중소·중견기업 지원"",""반도체 등 첨단산업 육성"",""물류비 지원"",""해외 전시회 및 수출 마케팅 지원""],""positive_reasons"":""무역금융 확대는 자금 조달 여력이 제한적인 수출기업의 생산·수주·운전자금 부담을 완화해 수출 물량 확대에 도움을 줄 수 있다. 삼성전자와 SK하이닉스는 정부의 반도체 중심 첨단산업 육성 전략의 직접적인 정책 수혜 가능성이 있으며, 현대차·기아·LG전자 등 글로벌 

pydantic.BaseModel.model_dump() -> dict  
pydantic.BaseModel.model_dump_json() -> json_str

In [ ]:
# result_json 컬럼 결측치 제거 / 인덱스 재정렬 
df = df.dropna(subset=['result_json'])
df = df.reset_index(drop=True)
df.head()

,date,category,press,title,document,link,summary,content,result,result_json
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,https://n.news.naver.com/mnews/article/052/0001759333?sid=101,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다.",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,"stock_related=True summary='정부는 원자재 가격 상승과 물류난 등 대외 리스크에 대응하고 수출 증가세를 유지하기 위해 하반기 수출지원 대책을 추진한다. 수출 중소·중견기업 지원을 위해 무역금융을 당초 계획보다 40조원 늘려 총 301조원 규모로 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대를 시행한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 방안을 마련해 수출 경쟁력과 무역수지 개선을 도모할 계획이다. 정책의 직접 수혜는 수출 중소기업과 첨단산업에 집중될 가능성이 높지만, 무역금융 확대만으로 수입 원자재 가격 상승에 따른 무역수지 적자가 해소된다고 보기는 어렵다.' positive_stocks=['삼성전자', 'SK하이닉스', '현대차', '기아', 'LG전자'] positive_keywords=['무역금융 40조원 확대', '수출 중소·중견기업 지원', '반도체 등 첨단산업 육성', '물류비 지원', '해외 전시회 및 수출 마케팅 지원'] positive_reasons='무역금융 확대는 자금 조달 여력이 제한적인 수출기업의 생산·수주·운전자금 부담을 완화해 수출 물량 확대에 도움을 줄 수 있다. 삼성전자와 SK하이닉스는 정부의 반도체 중심 첨단산업 육성 전략의 직접적인 정책 수혜 가능성이 있으며, 현대차·기아·LG전자 등 글로벌 판매 비중이 높은 기업도 금융·물류 지원에 따른 수출 비용 절감과 해외 마케팅 확대의 간접 수혜가 예상된다. 다만 대형 기업은 중소기업보다 정책 지원의 직접 효과가 제한적이고, 실제 주가 영향은 반도체 수요와 원자재·환율·운임 등 시장 변수에 더 크게 좌우될 수 있다.' negative_stocks=['HMM', '대한해운'] negative_keywords=['임시선박 월 4척 이상 투입', '중소기업 전용 선복 확대', '해상운임 안정화', '선복 공급 증가'] negative_reasons='정부가 임시선박을 지속 투입하고 중소기업 전용 선복을 확대하면 수출기업의 물류난은 완화되지만, 해상운송 공급이 늘어 운임이 하락할 경우 HMM·대한해운과 같은 해운사의 운임 및 수익성에는 부담이 될 수 있다. 다만 수출 물동량 증가가 운임 하락 효과를 상쇄할 가능성이 있어 부정적 영향은 해상운임과 글로벌 물동량의 방향에 따라 달라진다.'","{""stock_related"":true,""summary"":""정부는 원자재 가격 상승과 물류난 등 대외 리스크에 대응하고 수출 증가세를 유지하기 위해 하반기 수출지원 대책을 추진한다. 수출 중소·중견기업 지원을 위해 무역금융을 당초 계획보다 40조원 늘려 총 301조원 규모로 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대를 시행한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 방안을 마련해 수출 경쟁력과 무역수지 개선을 도모할 계획이다. 정책의 직접 수혜는 수출 중소기업과 첨단산업에 집중될 가능성이 높지만, 무역금융 확대만으로 수입 원자재 가격 상승에 따른 무역수지 적자가 해소된다고 보기는 어렵다."",""positive_stocks"":[""삼성전자"",""SK하이닉스"",""현대차"",""기아"",""LG전자""],""positive_keywords"":[""무역금융 40조원 확대"",""수출 중소·중견기업 지원"",""반도체 등 첨단산업 육성"",""물류비 지원"",""해외 전시회 및 수출 마케팅 지원""],""positive_reasons"":""무역금융 확대는 자금 조달 여력이 제한적인 수출기업의 생산·수주·운전자금 부담을 완화해 수출 물량 확대에 도움을 줄 수 있다. 삼성전자와 SK하이닉스는 정부의 반도체 중심 첨단산업 육성 전략의 직접적인 정책 수혜 가능성이 있으며, 현대차·기아·LG전자 등 글로벌 

## 학습용 데이터셋 변환
- system
- user(human)
- assistant(ai)

In [ ]:
df['system'] = system_prompt

df = df.rename(columns={
    'content': 'user',
    'result_json': 'assistant'
})

df[['system', 'user', 'assistant']]

system  \
0    \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
1    \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
2    \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
3    \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
4    \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
..                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       ...   
995  \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n   
996  \n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을

In [ ]:
df[['system', 'user', 'assistant']].to_json(
    'train.json',
    orient = 'records',
    force_ascii = False,
    indent = 4
)

In [ ]:
import os
from datasets import Dataset

dataset = Dataset.from_pandas(df[['system', 'user', 'assistant']])
dataset.push_to_hub(
    'capybaraOh/naver-economy-news2stock',
    token = os.environ['HF_TOKEN']
)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock/commit/23130c94e67b3f67d5bd76c44ccab1f4be8cc6fd', commit_message='Upload dataset', commit_description='', oid='23130c94e67b3f67d5bd76c44ccab1f4be8cc6fd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock', endpoint='https://huggingface.co', repo_type='dataset', repo_id='capybaraOh/naver-economy-news2stock'), pr_revision=None, pr_num=None)